# EMBEDDING LAB — entraînement E2 / E3 sur Colab (EL-013)

Ce notebook **ne mesure jamais le test**. Il régénère les données à l'identique, vérifie leurs empreintes, lance les diagnostics puis les entraînements, et évalue les meilleurs checkpoints sur la **validation** uniquement.

Avant de commencer : *Exécution → Modifier le type d'exécution → GPU (T4)*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code et dépendances
La branche est figée par son nom ; le commit exact est relevé dans chaque run.

In [ ]:
BRANCH = 'el-012-013-training'
!git clone -q --branch $BRANCH https://github.com/AbdGhni/EMBEDDING-LAB.git /content/EMBEDDING-LAB
%cd /content/EMBEDDING-LAB
!git log --oneline -1
# torch est déjà installé sur Colab (version CUDA) : pip ne le remplace pas
!pip -q install -e '.[data,ml]'

## 2. Checkpoints sur Google Drive
Les poids (≈ 470 Mo par run) vont sur ton Drive, pas dans Git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/embedding_lab_checkpoints'
!mkdir -p $CKPT

## 3. Régénérer les benchmarks et vérifier qu'ils sont identiques à la référence
SUMM-RE (texte seul, révision figée) ≈ 5 min ; B-SYN déterministe ; B-REAL depuis les annotations versionnées. Les exemples d'entraînement (`data/train/`) sont **versionnés dans le dépôt** : ils ne sont pas recalculés ici, car l'extraction de négatifs par E1 sur GPU pourrait changer légèrement d'ordre.

In [ ]:
!python -I scripts/fetch_summre.py
!python -I scripts/make_bsyn.py > /dev/null
!python -I scripts/prepare_breal_tasks.py val test train
!python -I scripts/assemble_breal.py --splits train val test
!python -I scripts/verify_data.py

**Si `verify_data.py` affiche des DIFFÉRENCES, s'arrêter** et me renvoyer la sortie : un entraînement sur des données différentes ne serait pas comparable.

## 4. Diagnostics (ce n'est pas un entraînement)
Dimensions, perte initiale, sur-apprentissage volontaire d'un mini-lot (`overfit_ok` doit valoir `true`), temps par étape au batch 64.

In [ ]:
!python -I scripts/diagnose_training.py --device cuda --batch 64

## 5. Entraînements
E2 = négatifs difficiles ; E3 = négatifs aléatoires, **même budget** (mêmes requêtes, mêmes positifs, même nombre d'exemples, même graine, même composition des batchs). Trois graines si le temps le permet ; sinon garder au moins la graine 13 pour E2 et E3.

In [ ]:
for seed in [13, 42, 1234]:
    for variant in ['e2', 'e3']:
        !python -I scripts/train_e2.py --variant $variant --seed $seed --checkpoint-root $CKPT

## 6. Récupérer les résultats
Télécharger l'archive et me la renvoyer dans la conversation. Elle contient config, manifeste (commit, GPU, empreintes), courbes et métriques de validation — pas les poids.

In [ ]:
!zip -qr /content/runs_e2_e3.zip runs -x 'runs/*/checkpoints/*'
from google.colab import files
files.download('/content/runs_e2_e3.zip')